# 강의 06 · 실습 10 — 패턴 4 수퍼바이저 · (3) 변형

## 1. 문제상황

- 홍보팀 팀장은 사내 행사 안내문 초안을 팀원에게 시킵니다.
- 초안은 조사 담당이 넣을 항목을 고르고, 작성 담당이 초안을 쓰고, 검토 담당이 완결된 두 문장 이상인지와 날짜·장소·신청 방법·신청 마감일이 다 들어 있는지 확인하는 순서로 만들어집니다.
- 검토 담당이 반려하면 작성 담당이 지적을 반영해 다시 써야 하는데, 지금은 순서가 코드에 고정되어 있어 반려가 나도 처음부터 다시 돌리거나 손으로 고칩니다.
- 팀장이 원하는 것은 안내문 요청만 주면 팀장 역할이 담당들을 시키고, 반려가 나면 작성만 다시 시키고, 합격이 나면 스스로 멈추는 흐름입니다.

## 2. 문제와 목표

- **문제**: 검토가 반려해도 작성으로 돌아갈 길이 없습니다. 다음 에이전트를 정하는 판단이 코드의 엣지에만 있고, 검토 결과를 보고 판단할 단계가 없습니다.
- **목표**: 안내문 요청을 넣으면 supervisor가 끝난 에이전트 목록과 마지막 검토 결과를 보고 다음 에이전트를 정하고, 검토가 반려면 작성 에이전트를 다시 부른 뒤 검토를 다시 시키고, 합격이면 스스로 종료하는 처리 흐름을 만듭니다.
    - supervisor: 끝난 에이전트 목록과 마지막 검토 결과를 보고 다음 담당을 `Assign`으로 정합니다(진입 상한 6).
    - 에이전트 셋: research(항목 뽑기), write(첫 초안은 아주 짧게, 지적이 있으면 반영해 두 문장으로), review(검토).
    - 검토 결과: 구조화 출력 `Verdict` — 합격/반려와 고칠 점 한 문장.
- **목표 달성 여부의 판정 기준**: 안내문 요청을 입력했을 때, 첫 검토가 반려되고, 작성이 한 번 더 이루어진 뒤, 두 번째 검토가 합격하여 끝나는 순서를 실행 결과에서 확인합니다. 작성은 두 번 이루어져야 합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex10_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 규격 두 개를 정의합니다.**
    - 안내문 요청(`request`), 작업 기록(`log`), 초안(`draft`), 검토 결과(`verdict`), 지적(`feedback`), supervisor 진입 횟수(`step`) 키 여섯 개를 가지는 상태를 선언합니다.
    - `log`에는 리듀서를 걸지 않습니다.
    - 배정 규격 `Assign`은 `who` 값을 research·write·review·done 넷으로 제한합니다.
    - 검토 규격 `Verdict`는 `grade` 값을 합격·반려 둘로 제한하고, `feedback` 필드에 고칠 점 한 문장을 담습니다.
    - 진입 횟수 상한은 6으로 둡니다.
2. **supervisor 노드를 만듭니다.**
    - 진입 횟수가 상한에 닿으면 END로 갑니다.
    - 아니면 끝난 에이전트 목록과 마지막 검토 결과를 `Assign` 규격을 건 모델(`with_structured_output`)에 넣습니다.
    - 지침은 「research, write, review 순서로 시킨다. 검토 결과가 반려면 write를 다시 시키고 그 뒤 review를 다시 시킨다. 검토 결과가 합격이면 done」입니다.
    - 받은 값이 done이면 END로, 아니면 그 에이전트로 가는 `Command(goto=..., update=...)`를 돌려주며 `step`을 1 올립니다.
3. **에이전트 서브그래프 세 개를 만듭니다.**
    - research는 안내문에 넣을 항목 2개를 뽑아 `log`에 덧붙입니다.
    - write는 `feedback`이 비어 있으면 안내문 초안을 25자 이내 한 문장으로 아주 짧게 쓰고, `feedback`이 있으면 지적을 반영해 두 문장으로 다시 써서 `draft`에 넣고 `log`에 「write: …」 줄을 덧붙입니다.
    - review는 `Verdict` 규격을 건 모델로 초안이 완결된 문장 두 개 이상이고 행사 날짜·장소·신청 방법·신청 마감일 넷이 모두 있으면 합격, 아니면 반려로 판정해 `verdict`·`feedback`에 쓰고 `log`에 「review: 합격/반려」 줄을 덧붙입니다.
    - 세 에이전트는 각각 노드 하나짜리 서브그래프이며 부모와 같은 상태를 씁니다.
4. **그래프에 노드를 등록합니다.**
    - supervisor 함수와, 컴파일된 서브그래프 세 개를 research·write·review라는 이름으로 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 supervisor로 가는 고정 엣지와, 에이전트 세 개에서 supervisor로 돌아가는 고정 엣지를 추가합니다.
    - supervisor에서 나가는 엣지와 조건부 엣지는 추가하지 않습니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 행사 날짜·장소·신청 방법·신청 마감일이 적힌 안내문 요청과 빈 기록, 빈 초안, 빈 검토 결과, 빈 지적, 진입 횟수 0을 넣어 실행한 뒤, 최종 진입 횟수와 작업 기록 전체와 초안을 출력합니다.
    - 값(`REQUEST`, 안내문 요청)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - supervisor는 진입할 때 「[supervisor] 진입 … -> who=값」 줄을, 검토 서브그래프는 「[sub:review/check] 검토 -> 판정」 줄을, 작성 서브그래프는 「[sub:write/draft] 첫 초안 쓰기」 또는 「[sub:write/draft] 지적을 반영해 다시 쓰기」 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키와 배정·검토 규격을 선언합니다 | `class NoticeState(TypedDict)`, `class Assign(BaseModel)`, `class Verdict(BaseModel)` | 1 |
| ② 노드 함수 정의 | 검토 결과를 보고 에이전트를 정하는 supervisor와 에이전트 서브그래프를 만듭니다 | `Command(goto=..., update=...)`, `StateGraph(NoticeState).compile()` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 supervisor와 서브그래프 셋을 등록합니다 | `StateGraph(NoticeState)`, `add_node` | 4 |
| ④ 엣지 연결 | 에이전트에서 수퍼바이저로 돌아오는 순서만 고정합니다 | `add_edge` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 요청을 넣어 실행합니다 | `compile()`, `invoke()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from pydantic import BaseModel, Field   # Field — 규격 필드에 설명(description=…)을 달 때 씁니다

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료: 안내문 요청 REQUEST — 값을 그대로 씁니다
REQUEST = ("10월 24일(금) 잠실 보조경기장에서 열리는 사내 체육대회 안내문. "
           "신청은 사내 포털 행사 메뉴에서 10월 10일까지.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. 요청·기록·초안·진입 횟수 키 네 개에 검토 결과 `verdict`와 지적 `feedback`이 더해집니다. 검토 규격 `Verdict`가 있으므로 supervisor는 검토 에이전트의 문장을 해석하지 않고 `verdict` 값만 봅니다. `log`에는 리듀서를 걸지 않습니다. 리듀서가 없으므로 `log`에 덧붙이는 노드는 새 줄 하나가 아니라, 기존 `log` 목록에 새 줄을 더한 전체 목록을 돌려줍니다.

In [ ]:
# 여기에 단계 ①(상태 정의, 배정 규격 Assign과 검토 규격 Verdict 정의, 상한 상수)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- supervisor는 `Command(goto=..., update=...)`를 돌려줍니다. 프롬프트에 마지막 검토 결과를 함께 싣고, 지침에 반려 처리 규칙이 들어 있습니다.
- write는 첫 시도에서 일부러 아주 짧게 씁니다. 반려 경로가 실행 결과에 보이도록 만든 규칙입니다. 지적이 있으면 지적을 반영해 두 문장으로 다시 씁니다.
- review는 `Verdict` 규격을 건 모델을 부릅니다. 판정과 지적을 규격대로 받으므로, 답변 문장을 뒤져 합격 여부를 찾아내지 않습니다.
- 에이전트 세 개는 노드 하나짜리 서브그래프이며 부모와 같은 스키마를 씁니다.

In [ ]:
# 여기에 단계 ②(supervisor 노드와 에이전트 서브그래프 세 개 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 노드를 등록합니다. 에이전트 세 개는 함수가 아니라 컴파일된 서브그래프를 그대로 노드로 등록합니다. 서브그래프가 부모와 같은 상태 스키마를 쓰므로 별도 변환 없이 얹힙니다. 이 구조에서 서브에이전트는 서브그래프로 구현됩니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. START에서 supervisor로 가는 엣지와, 에이전트 세 개에서 supervisor로 돌아오는 엣지를 추가합니다. supervisor에서 에이전트로 나가는 엣지는 추가하지 않습니다. supervisor가 돌려주는 `Command(goto=...)`가 다음 노드를 정하기 때문입니다. 다음 에이전트를 정하는 판단이 supervisor 한 곳에만 있어 중앙 집중형이 성립합니다. 반려 뒤 write로 되돌아가는 길도 엣지가 아니라 supervisor의 `goto`입니다.

In [ ]:
# 여기에 단계 ④(엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 요청과 빈 값들을 넣으면 최종 상태가 돌아옵니다. 아래에서는 날짜·장소·신청 방법·신청 마감일이 적힌 사내 체육대회 안내문을 요청합니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. supervisor의 `who` 값이 research, write, review, write, review, done 순서로 출력됩니다. 첫 검토에서 반려가 나오고, supervisor가 write를 다시 부릅니다.
2. `[sub:review/check] 검토` 줄이 두 번 출력되며, 첫 번째는 반려, 두 번째는 합격입니다.
3. 최종 상태의 `verdict`는 합격이고, 작업 기록에 write 줄이 두 개 있습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. 첫 검토가 합격이면 첫 초안을 아주 짧게 쓰라는 지침이 write에 있는지, 반려 뒤 done으로 끝나면 supervisor 프롬프트에 마지막 검토 결과가 실려 있는지 단계 ②를 다시 봅니다.